# test_reactor_00 — Config Survey: Reactor Tubular 1D

**Propósito:** verificar todos los builders de configuración del reactor sin ejecutar ninguna integración ODE.

## Ejes de configuración del equipo

| Eje | Opciones |
|-----|----------|
| Modo de operación (BC) | `batch` · `cstr` · `prf` |
| Tipo de reactor | tubo vacío (`catalyst_config=None`) · lecho catalítico |
| Modelo de pared | sin shell_tube · con shell_tube (`wall_config`) |
| Fuente de calor | convección externa (`thermal_bc`) · inducción electromagnética (`induction_config`) |
| Transporte | constante (`constant`) · correlaciones (`correlation`) |
| Reacciones | ninguna · homogéneas · heterogéneas catalíticas |

**Caso de referencia:** síntesis de NH₃ (N₂ + 3H₂ → 2NH₃), lecho catalítico de hierro, nc=3 especies, N=10 celdas.

## Modos de operación — qué parámetros los determinan

| Modo | Flujo de gas | v_in | C_in / T_in | v_out | Uso típico |
|------|-------------|------|-------------|-------|------------|
| `batch` | No | 0 | — | 0 | Estudio cinético, reactor cerrado |
| `cstr` | Sí | prescrito | prescritos | continuidad molar | N=1, mezcla perfecta |
| `prf` | Sí | prescrito | prescritos | continuidad molar | N>1, flujo pistón |

**Nota:** `cstr` y `prf` tienen las mismas BC matemáticas. La diferencia (mezcla perfecta vs. gradiente axial) viene del número de celdas N y de la dispersión axial en `trans_config`, no de la BC.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import numpy as np

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.units.reactor.config.boundary_c  import build_boundary_c_config
from src.units.reactor.config.thermal_bc  import build_thermal_bc_config
from src.units.reactor.config.transport   import build_transport_config
from src.units.reactor.config.gas_props   import build_gas_prop_config
from src.units.reactor.config.wall_c      import build_wall_config
from src.units.reactor.config.initial_c   import build_initial_conditions
from src.solvers.runner_reactor            import _validate_reactor_params
from src.physics.transport.transfer_coefficients import compute_transfer_coefficients
from src.physics.mixture_gas import compute_gas_mixture_properties

# ── Rutas a bases de datos ────────────────────────────────────────────────────
GAS_DB   = str(ROOT / "materials" / "fluids"  / "gasdb.txt")
SOLID_DB = str(ROOT / "materials" / "solids"  / "soliddb.txt")

# ── Geometría de referencia ───────────────────────────────────────────────────
Di    = 0.05    # [m]   diámetro interno del reactor
Do    = 0.056   # [m]   diámetro externo (pared 3 mm SS316L)
L     = 1.0     # [m]   longitud del lecho
N     = 10      # [—]   número de celdas axiales
dz    = L / N   # [m]   tamaño de celda
Ai    = np.pi / 4 * Di**2   # [m²]  área transversal interna
Pi    = np.pi * Di           # [m]   perímetro interno
Po    = np.pi * Do           # [m]   perímetro externo
epsi  = 0.40    # [—]   fracción de vacíos del lecho catalítico

# ── Especies gaseosas — caso NH₃ ─────────────────────────────────────────────
# N₂ + 3H₂ → 2NH₃   (síntesis de amoniaco, catalizador de hierro promovido)
species = ["N2", "H2", "NH3"]   # orden fijo; índices con .index()
nc      = len(species)           # nc = 3

# ── Condiciones de referencia ─────────────────────────────────────────────────
T_REF   = 298.15   # [K]   temperatura de referencia entálpica
T_OPER  = 700.0    # [K]   temperatura de operación (427 °C, zona de reacción)
P_OPER  = 200.0    # [bar] presión de operación típica síntesis NH₃

print(f"Geometría: Di={Di*1e3:.0f} mm, L={L:.1f} m, N={N}, dz={dz*1e3:.0f} mm")
print(f"Especies:  {species}  (nc={nc})")
print(f"Vacíos:    epsi={epsi:.2f}")

# ── Helper para mostrar valores en tablas ────────────────────────────────────
def _v_str(v):
    """Formatea un valor para tabla: None, escalar, array o callable."""
    if v is None:
        return "—"
    if callable(v):
        return f"fn({v(T_OPER):.3g} @ {T_OPER:.0f}K)"
    arr = np.asarray(v)
    if arr.ndim == 0:
        return f"{float(arr):.4g}"
    if arr.ndim == 1 and arr.size <= 3:
        return "[" + ", ".join(f"{x:.4g}" for x in arr) + "]"
    return f"array{arr.shape} [{arr.flat[0]:.4g}...{arr.flat[-1]:.4g}]"

---
## TEST 1 — `build_boundary_c_config` (3 modos)

In [ ]:
# Composición de entrada: mezcla estequiométrica N₂:H₂ = 1:3 (sin NH₃ al inlet)
# Estequiometría: N₂ + 3H₂ → 2NH₃  →  y_N2 = 0.25, y_H2 = 0.75
y_feed = np.zeros(nc)
y_feed[species.index("N2")] = 0.25   # fracción molar N₂
y_feed[species.index("H2")] = 0.75   # fracción molar H₂
# NH₃ = 0.00 al inlet (producto ausente en la alimentación)

V_IN = 0.02    # [m/s]  velocidad superficial del gas de entrada al reactor
T_IN = 700.0   # [K]    temperatura del gas de entrada

bc_batch = build_boundary_c_config(mode="batch", n_comp=nc, P_out_bar=P_OPER)
bc_cstr  = build_boundary_c_config(mode="cstr",  n_comp=nc, P_out_bar=P_OPER, v_in=V_IN, T_in=T_IN, y_in=y_feed)
bc_prf   = build_boundary_c_config(mode="prf",   n_comp=nc, P_out_bar=P_OPER, v_in=V_IN, T_in=T_IN, y_in=y_feed)

print(f"{'Clave':<15} {'batch':>22} {'cstr':>22} {'prf':>22}")
print("-" * 83)
for key in ["mode", "P_out_bar", "v_in", "T_in", "y_in"]:
    print(f"{key:<15} {_v_str(bc_batch[key]):>22} {_v_str(bc_cstr[key]):>22} {_v_str(bc_prf[key]):>22}")

print("\n✓ build_boundary_c_config — 3 modos OK")

---
## TEST 2 — `build_thermal_bc_config` (4 modos)

In [ ]:
e_wall = (Do - Di) / 2   # [m]  espesor de pared

# ── Modo adiabático ───────────────────────────────────────────────────────────
tbc_adiab = build_thermal_bc_config(
    mode="adiabatic", Di=Di, Do=Do, e_wall=e_wall,
)

# ── Modo fixed_twall (pared a temperatura prescrita) ──────────────────────────
tbc_fixed = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_OPER,
    k_wall=16.3,   # [W/m/K]  SS316L a 700K
)

# ── Modo heatfluxwall (potencia de calentamiento prescrita) ───────────────────
# Caso típico: calentamiento externo por resistencias eléctricas o horno
Q_HEAT = 500.0   # [W]  potencia total de calentamiento uniforme al reactor
tbc_flux = build_thermal_bc_config(
    mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
    Qwall=Q_HEAT,
    k_wall=16.3,
)

# ── Modo ambient_htc (reactor en horno o ambiente) ────────────────────────────
tbc_htc = build_thermal_bc_config(
    mode="ambient_htc", Di=Di, Do=Do, e_wall=e_wall,
    h_ambi=10.0,     # [W/m²/K]  convección natural exterior
    T_ambi=298.15,   # [K]       temperatura ambiente
    k_wall=16.3,
)

# ── Tabla resumen ─────────────────────────────────────────────────────────────
modes_tbc = [("adiabatic", tbc_adiab), ("fixed_twall", tbc_fixed),
             ("heatfluxwall", tbc_flux), ("ambient_htc", tbc_htc)]

keys_tbc = ["mode", "T_wall", "Qwall", "h_ambi", "T_ambi", "k_wall"]
print(f"{'Clave':<16}", end="")
for name, _ in modes_tbc:
    print(f"{name:>18}", end="")
print()
print("-" * 88)
for key in keys_tbc:
    print(f"{key:<16}", end="")
    for _, cfg in modes_tbc:
        print(f"{_v_str(cfg[key]):>18}", end="")
    print()

print("\n✓ build_thermal_bc_config — 4 modos OK")

---
## TEST 3 — `build_transport_config` (constant + correlation con valores visibles)

In [ ]:
# ── Modo constant ─────────────────────────────────────────────────────────────
trans_const = build_transport_config(
    mode="constant", N=N, n_comp=nc,
    h_bed=150.0,    # [W/m²/K]  HTC gas-catalizador prescrito
    h_wall=25.0,    # [W/m²/K]  HTC gas-pared prescrito
)

# ── Modo correlation ──────────────────────────────────────────────────────────
trans_corr = build_transport_config(
    mode="correlation", N=N, n_comp=nc,
    Pe_particle=2.0,
)

# Tabla modo constant
print("=== Modo constant ===")
for key in ["mode", "h_bed", "h_wall", "D_disp"]:
    print(f"  {key:<15}: {_v_str(trans_const[key])}")

# Valores de correlación visibles: calcular h_bed y h_wall a T_nodes
# Requiere prop_gas y gas_props → se calculan aquí para TEST 3
print("\n=== Modo correlation — valores a T_nodes ===")
prop_gas_t3 = build_gas_prop_config(species=species, mode="polynomial", db_path=GAS_DB)

# Perfil de temperatura representativo para el reactor de NH₃
T_nodes = np.linspace(650.0, 750.0, N)   # [K]  perfil Tg típico axial

# Composición media (entrada estequiométrica, antes de conversión apreciable)
y_mean     = np.tile(y_feed[:, None], (1, N))   # (nc, N)
x_mean     = y_mean.T                            # (N, nc) para Wilke
P_Pa_nodes = np.full(N, P_OPER * 1e5)           # [Pa]

gas_props_t3 = compute_gas_mixture_properties(
    P_Pa=P_Pa_nodes, Tg=T_nodes, x=x_mean,
    prop_gas=prop_gas_t3, n_comp=nc, N=N,
)

dp_cat = 3e-3   # [m]  diámetro de partícula de catalizador de hierro
a_p    = 6.0 * (1.0 - epsi) / dp_cat   # [m²/m³_bed] superficie específica

v_ref  = V_IN / epsi   # [m/s]  velocidad intersticial de referencia
u_rel  = np.full(N, v_ref)

trans_props_corr = compute_transfer_coefficients(
    Tg=T_nodes, Ts=T_nodes,   # Ts=Tg para el cálculo de referencia
    x=x_mean, gas_props=gas_props_t3,
    u_rel=u_rel,
    prop_gas=prop_gas_t3,
    prop_lecho={"D_p": np.full(N, dp_cat), "a_surf": np.full(N, a_p)},
    Di=Di, trans_config=trans_corr,
    n_comp=nc, N=N,
)

h_bed_corr  = trans_props_corr["h_bed"]
h_wall_corr = trans_props_corr["h_wall"]

print(f"  h_bed  (correlación) : min={h_bed_corr.min():.1f}  max={h_bed_corr.max():.1f}  [W/m²/K]")
print(f"  h_wall (correlación) : min={h_wall_corr.min():.1f}  max={h_wall_corr.max():.1f}  [W/m²/K]")
print(f"  D_disp               : {_v_str(trans_props_corr['D_disp'])}")

print("\n✓ build_transport_config — constant + correlation OK")

---
## TEST 4 — `build_gas_prop_config` (propiedades del gas)

In [ ]:
# ── Modo polynomial (recomendado para simulación) ─────────────────────────────
prop_gas = build_gas_prop_config(
    species=species,
    mode="polynomial",
    db_path=GAS_DB,
)

# ── Propiedades invariantes (no dependen de T) ────────────────────────────────
MW  = np.asarray(prop_gas["MW"]) * 1e3   # [g/mol] para presentación
print("Propiedades invariantes:")
print(f"  {'Especie':<8} {'MW [g/mol]':>12} {'Tmin [K]':>10} {'Tmax [K]':>10}")
print("  " + "-" * 42)
for i, sp in enumerate(species):
    print(f"  {sp:<8} {MW[i]:>12.3f} {prop_gas['Tref'][i]:>10.1f} {prop_gas['Tmax'][i]:>10.1f}")

# ── Propiedades evaluadas a T_OPER ────────────────────────────────────────────
print(f"\nPropiedades a T = {T_OPER:.0f} K:")
print(f"  {'Especie':<8} {'µ [µPa·s]':>12} {'k [mW/m/K]':>12} {'Cp [J/mol/K]':>14} {'h [J/mol]':>12}")
print("  " + "-" * 60)
for i, sp in enumerate(species):
    mu_i  = prop_gas["mu"][i](T_OPER) * 1e6
    k_i   = prop_gas["k"][i](T_OPER) * 1e3
    cp_i  = prop_gas["Cp_molar"][i](T_OPER)
    h_i   = prop_gas["h_molar"][i](T_OPER)
    print(f"  {sp:<8} {mu_i:>12.3f} {k_i:>12.3f} {cp_i:>14.2f} {h_i:>12.1f}")

print("\n✓ build_gas_prop_config — modo polynomial OK")

---
## TEST 5 — Params completo + validación + layout sv0

In [ ]:
# ── Configuración del catalizador genérico ────────────────────────────────────
# Valores representativos de un catalizador de lecho empaquetado industrial.
# En el test de NH₃ real se usarán propiedades del catalizador de hierro promovido.
dp_cat     = 3.0e-3    # [m]         diámetro de partícula de catalizador
rho_cat    = 800.0     # [kg/m³_bed] densidad bulk del catalizador en el lecho
Cp_cat_val = 500.0     # [J/kg/K]    capacidad calorífica del catalizador
a_p        = 6.0 * (1.0 - epsi) / dp_cat   # [m²/m³_bed] superficie específica

catalyst_config = {
    "dp":       dp_cat,
    "rho_bulk": rho_cat,
    "Cp_fn":    lambda T: np.full_like(np.asarray(T, float), Cp_cat_val),
    "a_p":      a_p,   # pre-calculado; el runner lo recalcula si falta
}

params = {
    "n_comp": nc, "N": N, "dz": dz, "Ai": Ai, "Di": Di, "Pi": Pi, "Po": Po,
    "prop_gas":   prop_gas,
    "MW":         np.asarray(prop_gas["MW"]),
    "gas_T_ref":  T_REF,
    "species":    species,
    "bc_config":         bc_prf,
    "thermal_bc_config": tbc_adiab,
    "trans_config":      trans_const,
    "epsi":             epsi,
    "catalyst_config":  catalyst_config,
    "reactions_config": [],   # sin reacciones en el survey
    "induction_config": None,
    "energy":           True,
}

_validate_reactor_params(params)
print("✓ _validate_reactor_params — params completo OK")

ic = build_initial_conditions(
    P_bar=P_OPER, Tg=T_OPER, y=y_feed,
    n_comp=nc, N=N,
    prop_gas=prop_gas, epsi=epsi, gas_T_ref=T_REF,
    Ts=T_OPER,   # catalizador inicialmente en equilibrio térmico con el gas
    Tw=None,
)
sv0 = ic["sv0"]
print(f"\nsv0.shape = {sv0.shape}   (esperado: {(nc+2)*N} = (nc+2)·N = ({nc}+2)·{N})")

In [ ]:
# ── Layout explícito de sv0 ───────────────────────────────────────────────────
# Con catalizador, sin shell_tube: [C(nc,N), Hg(N), Ts(N)] → (nc+2)·N
print("Layout del vector de estado sv0 (con catalizador, sin shell_tube):")
print(f"  Bloque          | índices sv        | shape    | valor c0")
print("  " + "-" * 64)

idx = 0
for i, sp in enumerate(species):
    seg = sv0[idx: idx + N]
    print(f"  C_{sp:<5}         | [{idx:>3}:{idx+N:<4}]        | ({N},)    | {seg[0]:.4g} mol/m³_gas")
    idx += N

seg_Hg = sv0[idx: idx + N]
print(f"  Hg              | [{idx:>3}:{idx+N:<4}]        | ({N},)    | {seg_Hg[0]:.4g} J/m³_bed")
idx += N

seg_Ts = sv0[idx: idx + N]
print(f"  Ts              | [{idx:>3}:{idx+N:<4}]        | ({N},)    | {seg_Ts[0]:.4g} K")
idx += N

print(f"\n  Total = {idx} = (nc+2)·N = ({nc}+2)·{N} ✓")

# ── Verificación de layout: C_N2 y Ts en la primera celda ────────────────────
C_N2_c0 = ic["C_init"][species.index("N2"), 0]
Tg_c0   = ic["Tg_init"][0]
Ts_c0   = ic["Ts_init"][0]
P_c0    = ic["P_init"][0]

print("\nValores iniciales en celda 0:")
print(f"  C_N2(0)  = {C_N2_c0:.4g}  mol/m³_gas")
print(f"  Tg(0)    = {Tg_c0:.1f}    K")
print(f"  Ts(0)    = {Ts_c0:.1f}    K")
print(f"  P(0)     = {P_c0:.1f}    bar")

# Verificar coherencia gas ideal: C = y·P/(R·T)
R_GAS = 8.31446261815324
C_N2_teorico = y_feed[species.index("N2")] * P_OPER * 1e5 / (R_GAS * T_OPER)
error_rel    = abs(C_N2_c0 - C_N2_teorico) / C_N2_teorico
print(f"\n  C_N2 ley gas ideal = {C_N2_teorico:.4g}  mol/m³_gas")
print(f"  Error relativo     = {error_rel:.2e}   {'✓ OK' if error_rel < 1e-8 else '✗ ERROR'}")

In [ ]:
params_tubo = {
    **params,
    "catalyst_config":  None,
    "epsi":             1.0,     # tubo vacío: todo vacíos
    "bc_config":        bc_batch,
}

_validate_reactor_params(params_tubo)
print("✓ _validate_reactor_params — caso tubo vacío OK")

ic_tubo = build_initial_conditions(
    P_bar=P_OPER, Tg=T_OPER, y=y_feed,
    n_comp=nc, N=N,
    prop_gas=prop_gas, epsi=1.0, gas_T_ref=T_REF,
)
sv0_tubo = ic_tubo["sv0"]
print(f"sv0_tubo.shape = {sv0_tubo.shape}   (esperado: {(nc+1)*N} = (nc+1)·N = ({nc}+1)·{N})")

print("\nLayouts de sv0 por configuración:")
print(f"  Sin catalizador, sin shell_tube : (nc+1)·N = {(nc+1)*N}")
print(f"  Con catalizador, sin shell_tube : (nc+2)·N = {(nc+2)*N}  ← este test")
print(f"  Con catalizador, con shell_tube : (nc+3)·N = {(nc+3)*N}")

print("\n✓ TEST 5 — Params completo + validación + layout sv0 OK")

---
## Resumen del config survey

| Builder | Modos verificados | Estado |
|---------|-------------------|--------|
| `build_boundary_c_config` | batch, cstr, prf | ✓ |
| `build_thermal_bc_config` | adiabatic, fixed_twall, heatfluxwall, ambient_htc | ✓ |
| `build_transport_config`  | constant, correlation (h_bed/h_wall visibles) | ✓ |
| `build_gas_prop_config`   | polynomial (N₂, H₂, NH₃) | ✓ |
| `_validate_reactor_params` | lecho catalítico + tubo vacío | ✓ |
| `build_initial_conditions` | con/sin catalizador, sv0 shape verificado | ✓ |

**Sin `run_step()` en ninguna celda.** El equipo está listo para los tests de integración.